# Semaine 1 - Préparation et Structuration des Données Médicales

**Objectif** : Collecter, nettoyer, anonymiser et structurer un corpus médical bilingue (FR/EN) pour le fine-tuning SFT et l'alignement DPO.

## Pipeline

| Étape | Description | Sortie |
|-------|-------------|--------|
| 1 | Téléchargement des données brutes | `data/raw/*.jsonl` |
| 2 | Anonymisation RGPD (Presidio) | `data/raw/*.jsonl` (écrasé) + rapport |
| 3 | Nettoyage + métadonnées | `data/processed/*.jsonl` |
| 4 | Formatage SFT/DPO + splits train/val/test | `data/sft/` + `data/dpo/` |

## Sources de données

| Dataset | Langue | Usage | Lien |
|---------|--------|-------|------|
| MediQAl | FR | SFT | [ANR-MALADES/MediQAl](https://huggingface.co/datasets/ANR-MALADES/MediQAl) |
| FrenchMedMCQA | FR | SFT | [qanastek/frenchmedmcqa](https://huggingface.co/datasets/qanastek/frenchmedmcqa) |
| MedQuAD | EN | SFT | [lavita/MedQuAD](https://huggingface.co/datasets/lavita/MedQuAD) |
| UltraMedical-Preference | EN | DPO | [TsinghuaC3I/UltraMedical-Preference](https://huggingface.co/datasets/TsinghuaC3I/UltraMedical-Preference) |

In [ ]:
import json
import hashlib
import logging
import random
from pathlib import Path
from collections import Counter

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Racine du projet
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Créer les dossiers
for d in ["raw", "processed", "sft", "dpo"]:
    (DATA_DIR / d).mkdir(parents=True, exist_ok=True)

print(f"Projet : {PROJECT_ROOT}")
print(f"Données : {DATA_DIR}")

In [ ]:
# Utilitaire de sauvegarde/lecture JSONL

def save_jsonl(data: list[dict], path: Path) -> None:
    """Sauvegarde une liste de dicts en JSONL."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"  -> {len(data)} exemples sauvegardés dans {path.name} ({path.stat().st_size / 1024:.0f} Ko)")


def load_jsonl(path: Path) -> list[dict]:
    """Charge un fichier JSONL."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

---
## Étape 1 : Téléchargement des données brutes

On télécharge chaque dataset depuis HuggingFace et on le sauvegarde **tel quel** en JSONL dans `data/raw/`.

Aucune transformation à ce stade — on conserve les données originales pour la traçabilité.

In [ ]:
# 1a. MediQAl — QCM médicaux francophones (ANR-MALADES)
# 3 configs : mcqu (QCM unique), mcqm (QCM multiple), oeq (questions ouvertes)

for config_name in ["mcqu", "mcqm"]:
    ds = load_dataset("ANR-MALADES/MediQAl", config_name, trust_remote_code=True, split="train")
    save_jsonl([dict(row) for row in ds], DATA_DIR / "raw" / f"mediqal_{config_name}.jsonl")

# oeq n'a qu'un split "test"
ds_oeq = load_dataset("ANR-MALADES/MediQAl", "oeq", trust_remote_code=True, split="test")
save_jsonl([dict(row) for row in ds_oeq], DATA_DIR / "raw" / "mediqal_oeq.jsonl")

print(f"\nMediQAl : mcqu={10113}, mcqm={5767}, oeq={len(ds_oeq)} exemples")

In [ ]:
# 1b. FrenchMedMCQA — QCM pharmacie francophones

ds = load_dataset("qanastek/frenchmedmcqa", trust_remote_code=True, split="train")
save_jsonl([dict(row) for row in ds], DATA_DIR / "raw" / "frenchmedmcqa.jsonl")
print(f"FrenchMedMCQA : {len(ds)} exemples")

In [ ]:
# 1c. MedQuAD — Q&A médicales anglophones

ds = load_dataset("lavita/MedQuAD", trust_remote_code=True, split="train")
save_jsonl([dict(row) for row in ds], DATA_DIR / "raw" / "medquad.jsonl")
print(f"MedQuAD : {len(ds)} exemples")

In [ ]:
# 1d. UltraMedical-Preference — paires préférentielles pour DPO

ds = load_dataset("TsinghuaC3I/UltraMedical-Preference", trust_remote_code=True, split="train")
save_jsonl([dict(row) for row in ds], DATA_DIR / "raw" / "ultramedical_preference.jsonl")
print(f"UltraMedical-Preference : {len(ds)} exemples")

In [ ]:
# Vérification : inventaire des fichiers raw

print("=== Fichiers bruts téléchargés ===")
for f in sorted((DATA_DIR / "raw").glob("*.jsonl")):
    data = load_jsonl(f)
    print(f"  {f.name:40s} {len(data):>8d} exemples  ({f.stat().st_size / 1024 / 1024:.1f} Mo)")

---
## Étape 2 : Anonymisation RGPD avec Presidio

Même si ces datasets publics ne contiennent pas de vraies données patients, on applique systématiquement une anonymisation pour :
- Respecter les exigences RGPD du CHSA
- Démontrer le processus dans le POC
- Masquer tout nom/prénom pouvant apparaître dans les cas cliniques

**Outils** :
- `AnalyzerEngine` : détecte les entités sensibles (PERSON, PHONE_NUMBER, EMAIL, etc.)
- `AnonymizerEngine` : remplace les entités par des placeholders (`<PERSON>`, `<PHONE>`, etc.)
- Modèles spaCy : `fr_core_news_md` (français), `en_core_web_sm` (anglais)

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import logging

# Couper le warning "Entity MISC is not mapped" à la source
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "fr", "model_name": "fr_core_news_md"},
    ],
}

provider = NlpEngineProvider(nlp_configuration=nlp_config)
nlp_engine = provider.create_engine()

analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=["fr"])
anonymizer = AnonymizerEngine()

# Entités PII à détecter
ENTITIES_TO_DETECT = ["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "LOCATION"]

# Remplacement par des placeholders lisibles
OPERATORS = {
    "PERSON": OperatorConfig("replace", {"new_value": "<PERSONNE>"}),
    "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<TELEPHONE>"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
    "LOCATION": OperatorConfig("replace", {"new_value": "<LIEU>"}),
    "DEFAULT": OperatorConfig("replace", {"new_value": "<PII>"}),
}

print("Moteurs Presidio initialisés (FR uniquement)")

In [ ]:
import warnings
import logging

# Couper les warnings MISC de Presidio
warnings.filterwarnings("ignore")
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)
logging.getLogger("presidio_analyzer").setLevel(logging.ERROR)

def anonymize_text(text: str, language: str = "fr") -> tuple[str, list[dict]]:
    """
    Anonymise un texte et retourne le texte anonymisé + log des entités trouvées.
    """
    if not text or not isinstance(text, str):
        return text, []
    
    # Analyse : détection des entités
    results = analyzer.analyze(
        text=text,
        language=language,
        entities=ENTITIES_TO_DETECT,
        score_threshold=0.4,
    )
    
    # Log des détections
    detections = []
    for r in results:
        detections.append({
            "entity_type": r.entity_type,
            "original": text[r.start:r.end],
            "start": r.start,
            "end": r.end,
            "score": r.score,
        })
    
    # Anonymisation
    if results:
        anonymized = anonymizer.anonymize(
            text=text,
            analyzer_results=results,
            operators=OPERATORS,
        )
        return anonymized.text, detections
    
    return text, []


# Test rapide
test_text = "Le patient Jean Dupont, 45 ans, habitant à Lyon, se présente aux urgences avec des douleurs thoraciques."
anon_text, detections = anonymize_text(test_text, "fr")
print(f"Original  : {test_text}")
print(f"Anonymisé : {anon_text}")
print(f"Détections: {detections}")

In [ ]:
def get_text_fields(item: dict) -> list[str]:
    """Identifie les champs textuels à anonymiser dans un exemple."""
    text_fields = []
    for key, value in item.items():
        if isinstance(value, str) and len(value) > 10:
            text_fields.append(key)
    return text_fields


def anonymize_dataset(input_path: Path, language: str) -> dict:
    """
    Anonymise un fichier JSONL in-place.
    
    Returns:
        Rapport d'anonymisation (stats par type d'entité)
    """
    data = load_jsonl(input_path)
    all_detections = []
    entity_counts = Counter()
    
    for item in tqdm(data, desc=f"Anonymisation {input_path.name}"):
        for field in get_text_fields(item):
            anon_text, detections = anonymize_text(item[field], language)
            item[field] = anon_text
            for d in detections:
                entity_counts[d["entity_type"]] += 1
            all_detections.extend(detections)
    
    # Sauvegarder le fichier anonymisé (écrase le raw)
    save_jsonl(data, input_path)
    
    report = {
        "file": input_path.name,
        "language": language,
        "total_examples": len(data),
        "total_entities_detected": len(all_detections),
        "entities_by_type": dict(entity_counts),
    }
    
    print(f"  {input_path.name}: {len(all_detections)} entités détectées -> {dict(entity_counts)}")
    return report

In [ ]:
# Anonymisation des fichiers FR uniquement
# Les datasets EN (MedQuAD, UltraMedical) sont des Q&A génériques sans données patients réelles

FILE_LANGUAGES = {
    "mediqal_mcqu.jsonl": "fr",
    "mediqal_mcqm.jsonl": "fr",
    "mediqal_oeq.jsonl": "fr",
    "frenchmedmcqa.jsonl": "fr",
    # medquad.jsonl et ultramedical_preference.jsonl : EN, pas de données patients -> skip
}

anonymization_reports = []

print("=== Anonymisation RGPD (datasets FR) ===")
for filename, lang in FILE_LANGUAGES.items():
    filepath = DATA_DIR / "raw" / filename
    if filepath.exists():
        report = anonymize_dataset(filepath, lang)
        anonymization_reports.append(report)
    else:
        print(f"  SKIP {filename} (fichier absent)")

# Ajouter un rapport "non traité" pour les fichiers EN
for en_file in ["medquad.jsonl", "ultramedical_preference.jsonl"]:
    anonymization_reports.append({
        "file": en_file,
        "language": "en",
        "total_examples": "-",
        "total_entities_detected": 0,
        "entities_by_type": {},
        "note": "Dataset anglophone générique — pas de données patients, anonymisation non requise",
    })

# Sauvegarder le rapport d'anonymisation
report_path = DATA_DIR / "raw" / "anonymization_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(anonymization_reports, f, indent=2, ensure_ascii=False)

print(f"\nRapport sauvegardé : {report_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Graphique 1 : Entités détectées par fichier ---
files = []
entity_counts = []
for r in anonymization_reports:
    if r["total_entities_detected"] and r["total_entities_detected"] != 0:
        files.append(r["file"].replace(".jsonl", ""))
        entity_counts.append(r["total_entities_detected"])

if files:
    colors = plt.cm.Set2(np.linspace(0, 1, len(files)))
    bars = axes[0].barh(files, entity_counts, color=colors)
    axes[0].set_xlabel("Nombre d'entités détectées")
    axes[0].set_title("Entités sensibles détectées par dataset")
    for bar, count in zip(bars, entity_counts):
        axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     str(count), va="center", fontweight="bold")
else:
    axes[0].text(0.5, 0.5, "Aucune entité détectée", ha="center", va="center", fontsize=14)
    axes[0].set_title("Entités sensibles détectées par dataset")

# --- Graphique 2 : Répartition par type d'entité (agrégé) ---
all_entity_types = {}
for r in anonymization_reports:
    for etype, count in r.get("entities_by_type", {}).items():
        all_entity_types[etype] = all_entity_types.get(etype, 0) + count

if all_entity_types:
    labels = list(all_entity_types.keys())
    values = list(all_entity_types.values())
    colors_pie = plt.cm.Pastel1(np.linspace(0, 1, len(labels)))
    wedges, texts, autotexts = axes[1].pie(
        values, labels=labels, autopct="%1.0f%%", colors=colors_pie, startangle=90
    )
    axes[1].set_title("Répartition par type d'entité anonymisée")
else:
    axes[1].text(0.5, 0.5, "Aucune entité détectée", ha="center", va="center", fontsize=14)
    axes[1].set_title("Répartition par type d'entité")

plt.tight_layout()
plt.show()

# Résumé texte
total = sum(r.get("total_entities_detected", 0) for r in anonymization_reports if isinstance(r.get("total_entities_detected"), int))
print(f"\nTotal entités anonymisées : {total}")
for r in anonymization_reports:
    if r.get("note"):
        print(f"  {r['file']}: {r['note']}")

---
## Étape 3 : Nettoyage, structuration et métadonnées

On transforme les données brutes anonymisées en un format unifié avec métadonnées.

### Schéma des métadonnées

| Champ | Description |
|-------|-------------|
| `id` | Identifiant unique (source_config_index) |
| `source` | Nom du dataset HuggingFace d'origine |
| `langue` | `fr` ou `en` |
| `type_question` | `mcq_unique`, `mcq_multiple`, `open_question`, `preference` |
| `sujet_medical` | Spécialité médicale (si disponible) |
| `niveau_confiance` | Score de confiance de la réponse (1.0 pour MCQ, 0.9 pour open, variable pour DPO) |
| `instruction` | Question/prompt formaté (SFT) |
| `response` | Réponse formatée (SFT) |
| `prompt` / `chosen` / `rejected` | Champs DPO |

In [ ]:
# Constantes pour le formatage MCQ

ANSWER_KEYS = ["answer_a", "answer_b", "answer_c", "answer_d", "answer_e"]
ANSWER_LABELS = ["A", "B", "C", "D", "E"]

SYSTEM_PROMPT = "Vous etes un assistant medical specialise dans le triage des urgences."


def clean_text(text: str) -> str:
    """Nettoyage basique : espaces multiples, caractères de contrôle."""
    if not isinstance(text, str):
        return ""
    import re
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)  # Caractères de contrôle
    text = re.sub(r' +', ' ', text)  # Espaces multiples
    text = re.sub(r'\n{3,}', '\n\n', text)  # Sauts de ligne excessifs
    return text.strip()


def format_mcq_instruction(item: dict, with_clinical_case: bool = False) -> str:
    """Formate un QCM en instruction texte."""
    parts = []
    if with_clinical_case and item.get("clinical_case"):
        parts.append(f"Cas clinique : {clean_text(item['clinical_case'])}")
    parts.append(f"Question : {clean_text(item['question'])}")
    parts.append("\nChoix possibles :")
    for key, label in zip(ANSWER_KEYS, ANSWER_LABELS):
        answer = item.get(key)
        if answer:
            parts.append(f"  {label}. {clean_text(answer)}")
    return "\n".join(parts)


def format_mcq_response(item: dict) -> str:
    """Formate la réponse correcte d'un QCM."""
    correct = item.get("correct_answers", "")
    if isinstance(correct, list):
        # FrenchMedMCQA : indices 0-based
        correct_labels = [ANSWER_LABELS[i] for i in correct if i < len(ANSWER_LABELS)]
    else:
        # MediQAl : string "A" ou "A,C"
        correct_labels = [c.strip() for c in str(correct).split(",")]

    response_parts = []
    for label in correct_labels:
        idx = ANSWER_LABELS.index(label) if label in ANSWER_LABELS else -1
        if idx >= 0:
            answer_text = item.get(ANSWER_KEYS[idx], "")
            response_parts.append(f"{label}. {clean_text(answer_text)}")

    if response_parts:
        return "La reponse correcte est : " + " ; ".join(response_parts)
    return f"La reponse correcte est : {correct}"


print("Fonctions de formatage définies.")

In [ ]:
# 3a. Traitement de MediQAl (mcqu + mcqm + oeq)

sft_processed = []

# MCQ unique
data = load_jsonl(DATA_DIR / "raw" / "mediqal_mcqu.jsonl")
for i, item in enumerate(data):
    sft_processed.append({
        "id": f"mediqal_mcqu_{i:05d}",
        "source": "ANR-MALADES/MediQAl",
        "config": "mcqu",
        "langue": "fr",
        "type_question": "mcq_unique",
        "sujet_medical": item.get("medical_subject", "general"),
        "niveau_confiance": 1.0,
        "instruction": format_mcq_instruction(item, with_clinical_case=True),
        "response": format_mcq_response(item),
    })
print(f"MediQAl/mcqu : {len(data)} -> {len([x for x in sft_processed if x['config']=='mcqu'])} exemples")

# MCQ multiple
data = load_jsonl(DATA_DIR / "raw" / "mediqal_mcqm.jsonl")
for i, item in enumerate(data):
    sft_processed.append({
        "id": f"mediqal_mcqm_{i:05d}",
        "source": "ANR-MALADES/MediQAl",
        "config": "mcqm",
        "langue": "fr",
        "type_question": "mcq_multiple",
        "sujet_medical": item.get("medical_subject", "general"),
        "niveau_confiance": 1.0,
        "instruction": format_mcq_instruction(item, with_clinical_case=True),
        "response": format_mcq_response(item),
    })
print(f"MediQAl/mcqm : {len(data)} exemples")

# Questions ouvertes
data = load_jsonl(DATA_DIR / "raw" / "mediqal_oeq.jsonl")
for i, item in enumerate(data):
    question = clean_text(item.get("question", ""))
    answer = clean_text(item.get("answer", ""))
    if question and answer:
        instruction = question
        if item.get("clinical_case"):
            instruction = f"Cas clinique : {clean_text(item['clinical_case'])}\n\nQuestion : {question}"
        sft_processed.append({
            "id": f"mediqal_oeq_{i:05d}",
            "source": "ANR-MALADES/MediQAl",
            "config": "oeq",
            "langue": "fr",
            "type_question": "open_question",
            "sujet_medical": item.get("medical_subject", "general"),
            "niveau_confiance": 0.9,
            "instruction": instruction,
            "response": answer,
        })
print(f"MediQAl/oeq : {len(data)} exemples")

print(f"\nTotal SFT après MediQAl : {len(sft_processed)}")

In [ ]:
# 3b. Traitement de FrenchMedMCQA

data = load_jsonl(DATA_DIR / "raw" / "frenchmedmcqa.jsonl")
count_before = len(sft_processed)

for i, item in enumerate(data):
    n_correct = item.get("number_correct_answers", 0)
    qtype = "mcq_unique" if n_correct == 0 else "mcq_multiple"  # 0 = single answer dans ce dataset
    sft_processed.append({
        "id": f"frenchmedmcqa_{i:05d}",
        "source": "qanastek/frenchmedmcqa",
        "config": "default",
        "langue": "fr",
        "type_question": qtype,
        "sujet_medical": "pharmacie",
        "niveau_confiance": 1.0,
        "instruction": format_mcq_instruction(item),
        "response": format_mcq_response(item),
    })

print(f"FrenchMedMCQA : {len(sft_processed) - count_before} exemples ajoutés")
print(f"Total SFT : {len(sft_processed)}")

In [ ]:
# 3c. Traitement de MedQuAD

data = load_jsonl(DATA_DIR / "raw" / "medquad.jsonl")
count_before = len(sft_processed)

for i, item in enumerate(data):
    question = clean_text(item.get("question", ""))
    answer = clean_text(item.get("answer", ""))
    if question and answer:
        sft_processed.append({
            "id": f"medquad_{i:05d}",
            "source": "lavita/MedQuAD",
            "config": "default",
            "langue": "en",
            "type_question": "open_question",
            "sujet_medical": item.get("question_type", "general"),
            "niveau_confiance": 0.9,
            "instruction": question,
            "response": answer,
        })

print(f"MedQuAD : {len(sft_processed) - count_before} exemples ajoutés")
print(f"Total SFT : {len(sft_processed)}")

In [ ]:
# 3d. Traitement de UltraMedical-Preference (DPO)

data = load_jsonl(DATA_DIR / "raw" / "ultramedical_preference.jsonl")
dpo_processed = []

for i, item in enumerate(data):
    prompt = item.get("prompt", "")
    chosen = item.get("chosen", [])
    rejected = item.get("rejected", [])
    if prompt and chosen and rejected:
        dpo_processed.append({
            "id": f"ultramedical_{i:05d}",
            "source": "TsinghuaC3I/UltraMedical-Preference",
            "langue": "en",
            "type_question": "preference",
            "sujet_medical": "general",
            "niveau_confiance": 0.8,
            "label_type": item.get("label_type", ""),
            "prompt": clean_text(prompt),
            "chosen": chosen,
            "rejected": rejected,
        })

print(f"UltraMedical-Preference : {len(dpo_processed)} exemples DPO")

In [ ]:
# 3e. Déduplication par hash du contenu

def deduplicate(data: list[dict], key_field: str) -> list[dict]:
    """Supprime les doublons exacts basés sur le hash d'un champ."""
    seen = set()
    unique = []
    for item in data:
        h = hashlib.md5(item[key_field].encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(item)
    removed = len(data) - len(unique)
    if removed > 0:
        print(f"  {removed} doublons supprimés")
    return unique


print(f"Avant déduplication : SFT={len(sft_processed)}, DPO={len(dpo_processed)}")
sft_processed = deduplicate(sft_processed, "instruction")
dpo_processed = deduplicate(dpo_processed, "prompt")
print(f"Après déduplication : SFT={len(sft_processed)}, DPO={len(dpo_processed)}")

In [ ]:
# 3f. Sauvegarde des données processed

save_jsonl(sft_processed, DATA_DIR / "processed" / "sft_all.jsonl")
save_jsonl(dpo_processed, DATA_DIR / "processed" / "dpo_all.jsonl")

In [ ]:
# 3g. Statistiques visuelles du corpus

import matplotlib.pyplot as plt
import numpy as np

df_sft = pd.DataFrame(sft_processed)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- 1. Par source ---
source_counts = df_sft["source"].value_counts()
colors1 = plt.cm.Set2(np.linspace(0, 1, len(source_counts)))
bars = axes[0, 0].barh(
    [s.split("/")[-1] for s in source_counts.index],
    source_counts.values,
    color=colors1,
)
for bar, count in zip(bars, source_counts.values):
    axes[0, 0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                     str(count), va="center", fontweight="bold")
axes[0, 0].set_title("Exemples SFT par source")
axes[0, 0].set_xlabel("Nombre d'exemples")

# --- 2. Par langue ---
lang_counts = df_sft["langue"].value_counts()
colors2 = ["#5B9BD5", "#ED7D31"]
wedges, texts, autotexts = axes[0, 1].pie(
    lang_counts.values,
    labels=[f"{l.upper()} ({c})" for l, c in zip(lang_counts.index, lang_counts.values)],
    autopct="%1.1f%%",
    colors=colors2,
    startangle=90,
    textprops={"fontsize": 12},
)
axes[0, 1].set_title("Répartition par langue")

# --- 3. Par type de question ---
type_counts = df_sft["type_question"].value_counts()
colors3 = plt.cm.Pastel1(np.linspace(0, 1, len(type_counts)))
bars = axes[1, 0].bar(type_counts.index, type_counts.values, color=colors3)
for bar, count in zip(bars, type_counts.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                     str(count), ha="center", fontweight="bold")
axes[1, 0].set_title("Par type de question")
axes[1, 0].set_ylabel("Nombre d'exemples")
axes[1, 0].tick_params(axis="x", rotation=15)

# --- 4. Top 10 sujets médicaux ---
subject_counts = df_sft["sujet_medical"].value_counts().head(10)
colors4 = plt.cm.tab10(np.linspace(0, 1, len(subject_counts)))
bars = axes[1, 1].barh(subject_counts.index[::-1], subject_counts.values[::-1], color=colors4)
axes[1, 1].set_title("Top 10 sujets médicaux")
axes[1, 1].set_xlabel("Nombre d'exemples")

plt.suptitle(f"Corpus SFT — {len(sft_processed)} exemples", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"\nCorpus DPO : {len(dpo_processed)} paires de préférences (UltraMedical-Preference)")

---
## Étape 4 : Formatage SFT/DPO et splits train/val/test

On produit les fichiers finaux au format attendu par les trainers TRL :
- **SFT** : `{"messages": [{"role": ..., "content": ...}, ...]}` + métadonnées
- **DPO** : `{"prompt": ..., "chosen": [...], "rejected": [...]}` + métadonnées

Splits : **80% train / 10% val / 10% test**

In [ ]:
# 4a. Échantillonnage SFT à 5000 exemples

MAX_SFT_SAMPLES = 5000
SEED = 42

random.seed(SEED)

if len(sft_processed) > MAX_SFT_SAMPLES:
    sft_sampled = random.sample(sft_processed, MAX_SFT_SAMPLES)
    print(f"Échantillonnage : {len(sft_processed)} -> {MAX_SFT_SAMPLES} exemples")
else:
    sft_sampled = sft_processed.copy()
    print(f"Pas d'échantillonnage nécessaire : {len(sft_sampled)} exemples")

# Vérifier la répartition des langues dans l'échantillon
lang_counts = Counter(item["langue"] for item in sft_sampled)
print(f"Répartition par langue : {dict(lang_counts)}")

In [ ]:
# 4b. Formatage SFT au format ChatML (messages)

def to_sft_format(item: dict) -> dict:
    """Convertit un exemple processed en format SFT (ChatML) avec métadonnées."""
    return {
        "id": item["id"],
        "source": item["source"],
        "langue": item["langue"],
        "type_question": item["type_question"],
        "sujet_medical": item["sujet_medical"],
        "niveau_confiance": item["niveau_confiance"],
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["instruction"]},
            {"role": "assistant", "content": item["response"]},
        ],
    }


sft_final = [to_sft_format(item) for item in sft_sampled]
print(f"{len(sft_final)} exemples SFT formatés")
print(f"\nExemple :")
print(json.dumps(sft_final[0], indent=2, ensure_ascii=False)[:500] + "...")

In [ ]:
# 4c. Split train / val / test (80/10/10)

def split_dataset(data: list[dict], train_ratio=0.8, val_ratio=0.1, seed=42) -> tuple:
    """Split un dataset en train/val/test."""
    random.seed(seed)
    shuffled = data.copy()
    random.shuffle(shuffled)
    
    n = len(shuffled)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train = shuffled[:train_end]
    val = shuffled[train_end:val_end]
    test = shuffled[val_end:]
    
    print(f"  train: {len(train)} | val: {len(val)} | test: {len(test)}")
    return train, val, test


# Split SFT
print("SFT split :")
sft_train, sft_val, sft_test = split_dataset(sft_final)
save_jsonl(sft_train, DATA_DIR / "sft" / "train.jsonl")
save_jsonl(sft_val, DATA_DIR / "sft" / "eval.jsonl")
save_jsonl(sft_test, DATA_DIR / "sft" / "test.jsonl")

In [ ]:
# 4d. Split DPO

# Le DPO conserve les métadonnées mais garde le format prompt/chosen/rejected
def to_dpo_format(item: dict) -> dict:
    """Convertit un exemple processed en format DPO avec métadonnées."""
    return {
        "id": item["id"],
        "source": item["source"],
        "langue": item["langue"],
        "niveau_confiance": item["niveau_confiance"],
        "prompt": item["prompt"],
        "chosen": item["chosen"],
        "rejected": item["rejected"],
    }


dpo_final = [to_dpo_format(item) for item in dpo_processed]

print("DPO split :")
dpo_train, dpo_val, dpo_test = split_dataset(dpo_final)
save_jsonl(dpo_train, DATA_DIR / "dpo" / "train.jsonl")
save_jsonl(dpo_val, DATA_DIR / "dpo" / "eval.jsonl")
save_jsonl(dpo_test, DATA_DIR / "dpo" / "test.jsonl")

In [ ]:
# 4e. Vérification : aucun leak entre splits

def check_no_leak(train, val, test, key="id"):
    train_ids = {item[key] for item in train}
    val_ids = {item[key] for item in val}
    test_ids = {item[key] for item in test}
    
    assert train_ids.isdisjoint(val_ids), "LEAK : train ∩ val non vide !"
    assert train_ids.isdisjoint(test_ids), "LEAK : train ∩ test non vide !"
    assert val_ids.isdisjoint(test_ids), "LEAK : val ∩ test non vide !"
    print("  Aucun leak détecté entre train/val/test")


print("Vérification SFT :")
check_no_leak(sft_train, sft_val, sft_test)
print("Vérification DPO :")
check_no_leak(dpo_train, dpo_val, dpo_test)

---
## Résumé final et inventaire des fichiers

In [ ]:
# Résumé final — Vue d'ensemble des splits

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- SFT splits ---
sft_sizes = [len(sft_train), len(sft_val), len(sft_test)]
sft_labels = [f"Train\n({sft_sizes[0]})", f"Val\n({sft_sizes[1]})", f"Test\n({sft_sizes[2]})"]
colors_sft = ["#4CAF50", "#FFC107", "#F44336"]
axes[0].pie(sft_sizes, labels=sft_labels, autopct="%1.0f%%", colors=colors_sft,
            startangle=90, textprops={"fontsize": 12})
axes[0].set_title(f"Dataset SFT — {sum(sft_sizes)} exemples", fontsize=13, fontweight="bold")

# --- DPO splits ---
dpo_sizes = [len(dpo_train), len(dpo_val), len(dpo_test)]
dpo_labels = [f"Train\n({dpo_sizes[0]})", f"Val\n({dpo_sizes[1]})", f"Test\n({dpo_sizes[2]})"]
axes[1].pie(dpo_sizes, labels=dpo_labels, autopct="%1.0f%%", colors=colors_sft,
            startangle=90, textprops={"fontsize": 12})
axes[1].set_title(f"Dataset DPO — {sum(dpo_sizes)} exemples", fontsize=13, fontweight="bold")

plt.suptitle("Répartition finale des splits Train / Val / Test", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Tableau récapitulatif
print("\n" + "=" * 70)
print(f"{'Fichier':<45} {'Exemples':>10} {'Taille':>10}")
print("=" * 70)
for subdir in ["raw", "processed", "sft", "dpo"]:
    print(f"\n  data/{subdir}/")
    for f in sorted((DATA_DIR / subdir).glob("*.jsonl")):
        n = sum(1 for _ in open(f))
        size = f.stat().st_size / 1024 / 1024
        print(f"    {f.name:<41} {n:>10,} {size:>8.1f} Mo")
print("\n" + "=" * 70)
print("Pipeline terminé. Données prêtes pour SFT (Semaine 2) et DPO (Semaine 3).")